<a href="https://colab.research.google.com/github/kaluarachchiteshan/agromaster-risk-calculation-ml/blob/03-cleaning/notebooks/03_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 03 - Cleaning (proposal 4.3) | Group 41 AgroMaster
# Every rule here is fixed or looks only at earlier dates, so nothing is learned from the test period.
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/agromaster"
KEYS = ["state", "district", "market", "commodity", "variety", "grade"]   # one series = one combination
PRICES = ["min_price", "max_price", "modal_price"]

# ---------------- CONFIG ----------------
OUTLIER_FACTOR = 10     # modal price more than 10x above/below the series' recent median = entry error
OUTLIER_WINDOW = 30     # "recent" = previous 30 reports of the same series (past only)
SPREAD_FACTOR  = 10     # max more than 10x modal, or min less than modal/10 = entry error
MARKET_FACTOR  = 20     # modal price more than 20x above/below the same-day commodity median = entry error
# ----------------------------------------

df = pd.read_parquet(os.path.join(OUT_DIR, "agmarknet_subset.parquet"))
log = [("loaded", len(df))]
def step(name):
    """Record the row count after a cleaning step."""
    log.append((name, len(df)))
    print(f"{name:42s} rows left: {len(df):>10,}  (removed {log[-2][1] - len(df):,})")
print(f"Loaded {len(df):,} rows")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 7,919,229 rows


In [ ]:
# Step 1: standardise category names (trim and collapse spaces), then list look-alike names
for c in KEYS:
    before = df[c].nunique()
    df[c] = df[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True).astype("category")
    print(f"{c:10s} categories: {before:>5} -> {df[c].nunique():>5}")
for c in ["state", "commodity", "variety", "grade"]:
    names = pd.Series(df[c].cat.categories)
    key = names.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
    alike = names.groupby(key).agg(list); alike = alike[alike.str.len() > 1]
    print(f"Look-alike names in {c}:", alike.tolist() if len(alike) else "none")

# Step 2: exact duplicates
df = df.drop_duplicates(KEYS + ["date"] + PRICES); step("drop exact duplicates")

# Step 3: missing or non-positive prices (a price of 0 means "not recorded")
zmin, zmax = df["min_price"] <= 0, df["max_price"] <= 0
print("\nZero-price patterns:", {"min only": int((zmin & ~zmax).sum()), "max only": int((~zmin & zmax).sum()),
                                 "both": int((zmin & zmax).sum()), "modal": int((df["modal_price"] <= 0).sum())})
bad = df[PRICES].isna().any(axis=1) | (df[PRICES] <= 0).any(axis=1)
print("Share of rows with a zero price, by year (last 8 years, %):")
print((100 * bad.groupby(df["date"].dt.year).mean()).round(2).tail(8).to_string())
df = df[~bad]; step("drop missing or non-positive prices")

state      categories:    25 ->    25
district   categories:   321 ->   321
market     categories:   787 ->   787
commodity  categories:     5 ->     5
variety    categories:    31 ->    31
grade      categories:     2 ->     2
Look-alike names in state: none
Look-alike names in commodity: none
Look-alike names in variety: none
Look-alike names in grade: none
drop exact duplicates                      rows left:  7,919,229  (removed 0)

Zero-price patterns: {'min only': 8485, 'max only': 1957, 'both': 186990, 'modal': 0}
Share of rows with a zero price, by year (last 8 years, %):
date
2018    0.25
2019    0.13
2020    0.04
2021    0.03
2022    0.11
2023    0.06
2024    0.03
2025    0.02
drop missing or non-positive prices        rows left:  7,721,797  (removed 197,432)


In [ ]:
# Step 4: price consistency, min <= modal <= max
swap = df["min_price"] > df["max_price"]            # entered in the wrong order -> swap
df.loc[swap, ["min_price", "max_price"]] = df.loc[swap, ["max_price", "min_price"]].values
print("min/max swapped in", int(swap.sum()), "rows")
df = df[(df["modal_price"] >= df["min_price"]) & (df["modal_price"] <= df["max_price"])]
step("drop modal outside [min, max]")
wide = (df["max_price"] > SPREAD_FACTOR * df["modal_price"]) | (df["min_price"] < df["modal_price"] / SPREAD_FACTOR)
df = df[~wide]; step("drop implausible min/max spread")

# Step 5: one row per series per date (median of each price if a date is reported more than once)
n_dup = int(df.duplicated(KEYS + ["date"]).sum())
if n_dup:
    df = df.groupby(KEYS + ["date"], observed=True, sort=False)[PRICES].median().reset_index()
    df[PRICES] = df[PRICES].astype("float32")
step("aggregate same-date reports (median)")

# Step 6: suspected data-entry errors in the modal price
# Rule A (isolated spike): more than 10x away from BOTH the median of the previous 30 reports and the
#         median of the next 5 reports of the same series. A genuine crash stays low afterwards, so it is kept.
# Rule B (impossible level): more than 20x away from the same-day median of that commodity across all markets.
df = df.sort_values(KEYS + ["date"], kind="stable").reset_index(drop=True)
df["sid"] = df.groupby(KEYS, observed=True).ngroup()
grp = df.groupby("sid")["modal_price"]
prev_med = grp.transform(lambda s: s.shift(1).rolling(OUTLIER_WINDOW, min_periods=5).median())
next_med = grp.transform(lambda s: s[::-1].shift(1).rolling(5, min_periods=3).median()[::-1])
day_med  = df.groupby(["commodity", "date"], observed=True)["modal_price"].transform("median")

def far(ratio, k):
    """True where the price is more than k times above or below the reference."""
    return (ratio > k) | (ratio < 1 / k)

r_prev, r_next, r_day = df["modal_price"] / prev_med, df["modal_price"] / next_med, df["modal_price"] / day_med
rule_a = far(r_prev, OUTLIER_FACTOR) & (far(r_next, OUTLIER_FACTOR) | next_med.isna())
rule_b = far(r_day, MARKET_FACTOR)
flag = rule_a | rule_b
kept_crash = far(r_prev, OUTLIER_FACTOR) & ~flag      # large moves that persisted, so they are kept as genuine

print(f"\nRule A (isolated spike): {int(rule_a.sum()):,} | Rule B (impossible level): {int(rule_b.sum()):,} | both: {int((rule_a & rule_b).sum()):,}")
print(f"Large moves kept as genuine (10x from recent median but sustained): {int(kept_crash.sum()):,}")
print("\nFlagged per commodity (too high / too low):")
print(pd.crosstab(df.loc[flag, "commodity"], np.where(df.loc[flag, "modal_price"] > day_med[flag], "too high", "too low")).to_string())
cols = ["commodity", "market", "date", "modal_price", "prev_med", "next_med", "day_med"]
view = df.assign(prev_med=prev_med, next_med=next_med, day_med=day_med)
print("\nExamples removed:");  print(view.loc[flag, cols].sample(min(8, int(flag.sum())), random_state=0).to_string(float_format="%.0f"))
print("\nExamples kept as genuine:"); print(view.loc[kept_crash, cols].sample(min(8, int(kept_crash.sum())), random_state=0).to_string(float_format="%.0f"))
df = df[~flag].reset_index(drop=True); step("drop suspected entry errors")

min/max swapped in 0 rows
drop modal outside [min, max]              rows left:  7,721,796  (removed 1)
drop implausible min/max spread            rows left:  7,716,480  (removed 5,316)
aggregate same-date reports (median)       rows left:  7,716,480  (removed 0)

Rule A (isolated spike): 365 | Rule B (impossible level): 542 | both: 215
Large moves kept as genuine (10x from recent median but sustained): 805

Flagged per commodity (too high / too low):
col_0         too high  too low
commodity                      
Brinjal              8       93
Green Chilli        13      190
Onion               13       65
Potato              42      105
Tomato              15      148

Examples removed:
            commodity         market       date  modal_price  prev_med  next_med  day_med
3178318  Green Chilli          Patan 2015-04-10           30      3000       300     2550
2187426       Brinjal  Kanjirappally 2012-04-23           26      1300      3000     1100
4596796  Green Chilli    Lohian

In [ ]:
# Save the cleaned data and the cleaning log
df = df.drop(columns="sid")
for c in KEYS:
    df[c] = df[c].cat.remove_unused_categories()
df.to_parquet(os.path.join(OUT_DIR, "agmarknet_clean.parquet"), compression="zstd", index=False)

rep = pd.DataFrame(log, columns=["step", "rows"])
rep["removed"] = (-rep["rows"].diff()).fillna(0).astype(int)
rep["removed_%"] = (100 * rep["removed"] / rep["rows"].iloc[0]).round(3)
rep.to_csv(os.path.join(OUT_DIR, "cleaning_log.csv"), index=False)
print(rep.to_string(index=False))
print(f"\nKept {len(df):,} of {log[0][1]:,} rows ({100 * len(df) / log[0][1]:.2f}%) | series: {df.groupby(KEYS, observed=True).ngroups:,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()} | test rows (2024+): {(df['date'] > '2023-12-31').sum():,}")
print("\nPrice range after cleaning (INR per quintal):")
print(df.groupby("commodity", observed=True)[PRICES].agg(["min", "max"]).round(0).to_string())

                                step    rows  removed  removed_%
                              loaded 7919229        0      0.000
               drop exact duplicates 7919229        0      0.000
 drop missing or non-positive prices 7721797   197432      2.493
       drop modal outside [min, max] 7721796        1      0.000
     drop implausible min/max spread 7716480     5316      0.067
aggregate same-date reports (median) 7716480        0      0.000
         drop suspected entry errors 7715788      692      0.009

Kept 7,715,788 of 7,919,229 rows (97.43%) | series: 2,605
Date range: 2001-01-10 to 2025-11-06 | test rows (2024+): 782,981

Price range after cleaning (INR per quintal):
             min_price          max_price          modal_price         
                   min      max       min      max         min      max
commodity                                                              
Brinjal           25.0  30000.0      35.0  36000.0        30.0  30000.0
Green Chilli      60